# Notebook 10b - Training RGB Thresholds from ceilometer

In [19]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
import matplotlib.pyplot as plt
import xarray as xr
import os
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

In [77]:
path = '/storage/cdalden/goes/colorado/goes16/rgb_composite/'
file_template = 'combined_goes16_C02_C05_C13_rgb_colorado_{yearmonth}.nc'
datasets = {}
files = ['202205', '202206', '202207', '202208', '202301', '202302', '202304', '202305']

for file in files:
    datasets[file] = xr.open_dataset(path + file_template.format(yearmonth=file))
    print('opened {i}'.format(i=file))

if len(files) > 1:
    rgb_ds = xr.concat([datasets[file] for file in files], dim='t', combine_attrs='override')
else:
    rgb_ds = (datasets[files[0]])
rgb_ds = rgb_ds.rename({'t': 'time'})
rgb_ds
rgb_ds = rgb_ds.sel(latitude=38.958, longitude=-106.989, method='nearest')


opened 202205
opened 202206
opened 202207
opened 202208
opened 202301
opened 202302
opened 202304
opened 202305


In [99]:
start_time = pd.to_datetime('2022-08-01')
end_time = pd.to_datetime('2022-08-31')

In [100]:

# Select time range and filter after 14Z
rgb_ds = rgb_ds.sel(
    time=rgb_ds['time'].where(
        ((rgb_ds['time'] > start_time) &
        (rgb_ds['time'] < end_time) &
        (rgb_ds['time'].dt.hour >= 14))
    ).dropna('time')
)


In [101]:
path = '/storage/cdalden/goes/surface_obs/gucceilM1.b1/'
file = 'sail_guc_ceilometer_2021_2023.nc'
ceil_ds = xr.open_dataset(path + file)
# Select time range and filter between 8am local and 6pm local (MDT) in one step
ceil_ds = ceil_ds.sel(
    time=ceil_ds['time'].where(
        ((ceil_ds['time'] > start_time) &
        (ceil_ds['time'] < end_time) &
        (ceil_ds['time'].dt.hour >= 14))
    ).dropna('time')
)
ceil_binary = (~ceil_ds['first_cbh'].isnull()).astype(int)


In [110]:
# 1) Remove duplicate times if present
_, uniq_idx = np.unique(ceil_binary['time'], return_index=True)
ceil_binary = ceil_binary.isel(time=np.sort(uniq_idx))

# 2) Build offset 5-min target times (offset = 2.5 minutes = 150 seconds)
t0 = pd.to_datetime(ceil_binary['time'].values.min())
t1 = pd.to_datetime(ceil_binary['time'].values.max())

# Start at first 5-min multiple after t0, then add 2.5 min offset
# choose an anchored start: round up to nearest 5min boundary then add offset
start_5min = (t0.ceil('5min') + pd.Timedelta(seconds=150))
target_times = pd.date_range(start=start_5min, end=t1, freq='5min')

# 3) Pick the nearest original sample to each target time (tolerance = 2.5 min)
tol = pd.Timedelta(seconds=150)
ceil_binary_5min_offset = ceil_binary.sel(time=target_times, method='nearest', tolerance=tol)

# Optional: treat bins with no nearby sample as 0 (or keep NaN)
ceil_binary_5min_offset = ceil_binary_5min_offset.fillna(0).astype(int)

# 4) Quick checks
print("target times (first 6):", target_times[:6])
print("Result count:", ceil_binary_5min_offset.sizes['time'])
print("NaNs (before fill):", ceil_binary.sel(time=target_times, method='nearest', tolerance=tol).isnull().sum().item())
print("Sum (number of 1s):", int(ceil_binary_5min_offset.sum().item()))

# Replace original variable if desired
ceil_binary = ceil_binary_5min_offset

target times (first 6): DatetimeIndex(['2022-08-01 14:02:30', '2022-08-01 14:07:30',
               '2022-08-01 14:12:30', '2022-08-01 14:17:30',
               '2022-08-01 14:22:30', '2022-08-01 14:27:30'],
              dtype='datetime64[ns]', freq='5min')
Result count: 8471
NaNs (before fill): 0
Sum (number of 1s): 4600


In [111]:

# Select the nearest ceilometer binary for each RGB timestep
nearest_ceil = ceil_binary.sel(time=rgb_ds['time'], method='nearest')

# Assign to rgb_ds
rgb_ds['cloud_binary'] = nearest_ceil

# for testing, let's slice the training ds a little
# goes_ceil_ds = rgb_ds.sel(time=slice('2022-08-01', '2022-08-31'))
goes_ceil_ds = rgb_ds
# goes_ceil_ds = goes_ceil_ds.isel(time=slice(0, None, 6))

# Create a mask where all variables are non-NaN
mask = ~goes_ceil_ds.to_array().isnull().any(dim='variable')

# Apply the mask to filter the dataset
goes_ceil_ds = goes_ceil_ds.sel(time=goes_ceil_ds['time'][mask])

print(str(np.round(goes_ceil_ds.nbytes/10000,1)) + 'mb')

# drop lon and lat since they are constant
goes_ceil_ds = goes_ceil_ds.drop_vars(['latitude', 'longitude'])

goes_ceil_df = goes_ceil_ds.to_dataframe()
n_nans = nearest_ceil.isnull().sum().item()
rgb_ds

0.0mb


<xarray.Dataset> Size: 104kB
Dimensions:       (time: 2880)
Coordinates:
  * time          (time) datetime64[ns] 23kB 2022-08-01T14:02:30 ... 2022-08-...
    latitude      float64 8B 38.96
    longitude     float64 8B -107.0
Data variables:
    green         (time) float32 12kB 0.2263 0.213 0.192 ... 0.1633 0.1621
    blue          (time) float64 23kB 0.3091 0.3135 0.2628 ... 0.2773 0.3362
    red           (time) float64 23kB 0.3454 0.3412 0.3433 ... 0.0 0.0 0.0
    cloud_binary  (time) float64 23kB nan nan nan nan nan ... nan nan nan nan

In [106]:

# Prepare the data
# X = goes_tsi_ds[['red', 'green', 'blue']].to_dataframe().dropna()
# y = goes_tsi_ds['cloud_binary'].to_dataframe().loc[X.index]
X = goes_ceil_df[['red', 'green', 'blue']].dropna()
y = goes_ceil_df['cloud_binary'].loc[X.index]

# Split the data into training and testing sets
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Train the decision tree classifier
clf = DecisionTreeClassifier(max_depth=2, min_samples_split=10, class_weight='balanced', random_state=42)  # Limit depth for interpretability
clf.fit(X_train, y_train)

# Make predictions
y_pred = clf.predict(X_test)

# Calculate performance metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='binary')
recall = recall_score(y_test, y_pred, average='binary')
f1 = f1_score(y_test, y_pred, average='binary')

# Print the metrics
print("Performance Metrics:")
print(f"Accuracy: {accuracy:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1 Score: {f1:.2f}")

# Print a detailed classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['No Cloud', 'Cloud']))

# Visualize the decision tree
plt.figure(figsize=(12, 8))
# plot_tree(clf, feature_names=['red', 'green', 'blue'], class_names=['No Cloud', 'Cloud'], filled=True)
plt.show()

# Output the decision rules
tree_rules = export_text(clf, feature_names=['red', 'green', 'blue'])
print(tree_rules)

ValueError: With n_samples=0, test_size=0.3 and train_size=None, the resulting train set will be empty. Adjust any of the aforementioned parameters.